# Classificação e Rastreio da Depressão com Support Vector Machines (SVM)

Neste notebook, aplicamos o algoritmo **LinearSVC** para a classificação binária do NHIS 2025. O SVM busca o hiperplano de separação de máxima margem entre as classes, oferecendo uma abordagem matemática baseada em fronteira linear para comparar com os modelos de árvore (Random Forest e XGBoost).

## 1. Importação de Bibliotecas e Configuração dos Parâmetros Iniciais
Nesta etapa, carregamos os módulos essenciais do `scikit-learn` (incluindo o algoritmo `LinearSVC` e utilitários de pré-processamento), definimos as colunas que representam os sintomas diretos de depressão do PHQ-9 para remoção (*data leakage prevention*) e transformamos o target original multiclasse em uma abordagem **binária** (`0`: Sem Depressão, `1`: Com Depressão).

In [2]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Carregar a base limpa e aplicar a transformação binária
df = pd.read_csv('adult25_limpo.csv').set_index('ID_DOMICILIO')
df = df[~df['SEVERIDADE_DEPRESSAO_PHQ'].isin([-1, 5])].copy()
df['DEPRESSAO_PHQ_SIMPLIFICADO'] = (df['SEVERIDADE_DEPRESSAO_PHQ'] != 1).astype(int)

TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
PESO_COLUNA = 'PESO_AMOSTRAL'

COLUNAS_SINTOMAS = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ'
]



## 2. Seleção de Features e Filtragem Estrita de Respostas
Isolamos as variáveis preditoras ($X$), o target binário ($y$) e o peso amostral (`PESO_AMOSTRAL`). Aplicamos o filtro estrito (Cenário B) para remover registros com códigos de não-resposta (`-1`) nas variáveis preditoras, garantindo a integridade metodológica e evitando ruídos no treinamento.

In [3]:
# 2. Isolar X, y e Pesos (Apenas registros completos)
y_bin = df[TARGET_BIN]
weights = df[PESO_COLUNA]
X_base = df.drop(columns=[col for col in [TARGET_BIN, PESO_COLUNA] + COLUNAS_SINTOMAS if col in df.columns])

mascara_completos = (X_base != -1).all(axis=1)
X_bin = X_base[mascara_completos].copy()
y_bin = y_bin[mascara_completos].copy()
w_bin = weights[mascara_completos].copy()



## 3. Divisão Estratificada do Conjunto de Dados (Train / Test Split)
Dividimos a base de dados estrita em **80% para treino** e **20% para teste**. O uso do parâmetro `stratify=y_bin` garante que a proporção exata entre pessoas com e sem depressão seja preservada em ambos os subconjuntos, além de transportar o peso de expansão populacional (`w_tr` e `w_te`).

In [4]:
# 3. Split Estratificado
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
    X_bin, y_bin, w_bin, test_size=0.2, random_state=42, stratify=y_bin
)



## 4. Padronização Numérica e Codificação Categórica (Preprocessing)
Construímos o pipeline de transformação de dados (`ColumnTransformer`):
* **Normalização (`StandardScaler`):** Aplicada às variáveis contínuas (como `RAZAO_RENDA_POBREZA`) para colocar os valores na mesma escala (essencial para o cálculo de margem do SVM).
* **One-Hot Encoding (`OneHotEncoder`):** Transforma as variáveis categóricas sociodemográficas em vetores binários, descartando a primeira categoria para evitar multicolinearidade.

In [5]:
# 4. Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA']
cat_cols = [c for c in X_tr.columns if c not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_tr_proc = preprocessor.fit_transform(X_tr)
X_te_proc = preprocessor.transform(X_te)



## 5. Definição do Espaço de Busca de Hiperparâmetros
Configuramos a grade de hiperparâmetros para otimização do `LinearSVC`:
* **`C` (Parâmetro de Regularização):** Controla o *trade-off* entre obter uma margem mais larga e minimizar os erros de classificação.
* **`class_weight`:** Testamos o ajuste automático de pesos (`balanced` vs `None`) para lidar com o desbalanceamento das classes.
* **`tol` (Tolerância do Critério de Parada):** Controla a precisão de convergência do algoritmo.

In [6]:
# 5. Espaço de Busca de Hiperparâmetros para o LinearSVC
param_dist_svm = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0],
    'class_weight': ['balanced', None],
    'tol': [1e-4, 1e-3, 1e-2]
}



## 6. Otimização do SVM com Validação Cruzada (RandomizedSearchCV)
Executamos a busca estocástica com 10 combinações em 3 dobras de validação cruzada (`cv=3`), otimizando diretamente a métrica de $F1\text{-score}$. O algoritmo treina a fronteira de separação considerando o peso amostral populacional (`sample_weight`).

In [7]:
# 6. Otimização com RandomizedSearchCV
svm_base = LinearSVC(random_state=42, max_iter=3000, dual=False)

random_search_svm = RandomizedSearchCV(
    estimator=svm_base,
    param_distributions=param_dist_svm,
    n_iter=10,
    scoring='f1',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("Iniciando otimização do SVM (LinearSVC)...")
inicio = time.time()
random_search_svm.fit(X_tr_proc, y_tr, sample_weight=w_tr)
tempo_total = time.time() - inicio

svm_otimizado = random_search_svm.best_estimator_

print(f"\n✓ Otimização concluída em {tempo_total:.2f}s")
print("\n--- MELHORES HIPERPARÂMETROS ENCONTRADOS (SVM) ---")
for param, val in random_search_svm.best_params_.items():
    print(f"{param}: {val}")



Iniciando otimização do SVM (LinearSVC)...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

✓ Otimização concluída em 28.05s

--- MELHORES HIPERPARÂMETROS ENCONTRADOS (SVM) ---
tol: 0.01
class_weight: None
C: 1.0


## 7. Avaliação e Projeção Populacional do Modelo SVM
Avaliamos o modelo final no conjunto de teste independente:
1. **Função de Decisão (`decision_function`):** Calcula a distância das instâncias até o hiperplano de separação para computar o score ROC-AUC.
2. **Matrizes de Confusão Ponderadas:** Exibimos os números absolutos e percentuais de Verdadeiros Negativos (TN), Falsos Positivos (FP), Falsos Negativos (FN) e Verdadeiros Positivos (TP) tanto para a amostra local quanto para a projeção nacional expandida nos EUA.

In [8]:
# 7. Avaliação no Conjunto de Teste
y_pred_svm = svm_otimizado.predict(X_te_proc)
y_score_svm = svm_otimizado.decision_function(X_te_proc)

print("\n--- RELATÓRIO DE CLASSIFICAÇÃO SVM (PONDERADO) ---")
print(classification_report(y_te, y_pred_svm, sample_weight=w_te, zero_division=0))

# Quadrantes da Matriz de Confusão
cm_amostra = confusion_matrix(y_te, y_pred_svm)
tn, fp, fn, tp = cm_amostra.ravel()
total_amostra = len(y_te)

cm_pop = confusion_matrix(y_te, y_pred_svm, sample_weight=w_te)
tn_w, fp_w, fn_w, tp_w = cm_pop.ravel()
total_pop = w_te.sum()

print("--- DETALHAMENTO DA MATRIZ DE CONFUSÃO DA AMOSTRA (Linhas de Teste) ---")
print(f"Verdadeiros Negativos (TN): {tn:5d} | {tn/total_amostra*100:6.2f}% (Sem Depressão - Correto)")
print(f"Falsos Positivos      (FP): {fp:5d} | {fp/total_amostra*100:6.2f}% (Alarme Falso)")
print(f"Falsos Negativos      (FN): {fn:5d} | {fn/total_amostra*100:6.2f}% (Subdiagnóstico)")
print(f"Verdadeiros Positivos (TP): {tp:5d} | {tp/total_amostra*100:6.2f}% (Com Depressão - Capturado)")

print("\n--- DETALHAMENTO POPULACIONAL EXPANDIDO (Projeção Nacional) ---")
print(f"Verdadeiros Negativos (TN): {tn_w:11,.0f} | {tn_w/total_pop*100:6.2f}%")
print(f"Falsos Positivos      (FP): {fp_w:11,.0f} | {fp_w/total_pop*100:6.2f}%")
print(f"Falsos Negativos      (FN): {fn_w:11,.0f} | {fn_w/total_pop*100:6.2f}%")
print(f"Verdadeiros Positivos (TP): {tp_w:11,.0f} | {tp_w/total_pop*100:6.2f}%")

print(f"\nROC-AUC Score (SVM): {roc_auc_score(y_te, y_score_svm, sample_weight=w_te):.4f}")


--- RELATÓRIO DE CLASSIFICAÇÃO SVM (PONDERADO) ---
              precision    recall  f1-score   support

           0       0.90      0.94      0.92 37773204.546999976
           1       0.76      0.62      0.68 10425277.420000015

    accuracy                           0.88 48198481.96699999
   macro avg       0.83      0.78      0.80 48198481.96699999
weighted avg       0.87      0.88      0.87 48198481.96699999

--- DETALHAMENTO DA MATRIZ DE CONFUSÃO DA AMOSTRA (Linhas de Teste) ---
Verdadeiros Negativos (TN):  3308 |  74.89% (Sem Depressão - Correto)
Falsos Positivos      (FP):   174 |   3.94% (Alarme Falso)
Falsos Negativos      (FN):   353 |   7.99% (Subdiagnóstico)
Verdadeiros Positivos (TP):   582 |  13.18% (Com Depressão - Capturado)

--- DETALHAMENTO POPULACIONAL EXPANDIDO (Projeção Nacional) ---
Verdadeiros Negativos (TN):  35,692,631 |  74.05%
Falsos Positivos      (FP):   2,080,573 |   4.32%
Falsos Negativos      (FN):   3,924,843 |   8.14%
Verdadeiros Positivos (TP):   

### Análise Crítica dos Resultados: Support Vector Machines (LinearSVC)

* **Acurácia Global (88%) e Alta Precisão (76%):** O SVM apresentou a maior acurácia global (88%) e a maior precisão para a classe positiva (76%) entre todos os modelos testados. Isso se traduz em uma taxa excepcionalmente baixa de falsos alarmes (falsos positivos de apenas 4,32% na população).
* **Impacto na Sensibilidade (Recall = 62%):** Como a busca de hiperparâmetros selecionou `class_weight: None`, a margem do hiperplano priorizou a separação límpida da classe majoritária. Consequentemente, o *Recall* situou-se em 62%, deixando uma margem maior de subdiagnóstico (~3,92 milhões de indivíduos) quando comparado ao XGBoost (76%) e ao Random Forest (71,8%).
* **Capacidade Discriminativa (ROC-AUC = 0.9109):** O score ROC-AUC obtido a partir da função de decisão confirma que as variáveis sociodemográficas oferecem uma separação linear consistente em alta dimensão.
* **Conclusão Comparativa:** O SVM destaca-se pela **eficiência e precisão**, sendo ideal para cenários onde a prioridade é evitar diagnósticos falso-positivos.

### Benchmark Comparativo dos Modelos Binários

| Métrica Populacional | Random Forest (Otimizado) | XGBoost (Otimizado) | SVM (LinearSVC) | Destaque Prático |
| :--- | :---: | :---: | :---: | :--- |
| **ROC-AUC Score** | **0.9135** | 0.9092 | 0.9109 | Random Forest lidera em discriminação |
| **Acurácia Global** | 86% | 86% | **88%** | **SVM possui a maior acurácia geral** |
| **Precisão (Classe 1)** | 67% | 66% | **76%** | **SVM gera menos falsos alarmes** |
| **Recall / Sensibilidade** | 71.76% | **76.00%** | 62.00% | **XGBoost captura mais casos reais** |
| **F1-Score (Classe 1)** | 0.70 | **0.71** | 0.68 | XGBoost obtém o melhor equilíbrio |